# Tartu Airport (EETU) Arrivals

Proof of concept: compare two FlightAware AeroAPI endpoints for arrivals into
Tartu Airport:

- `GET /airports/{id}/flights/arrivals` — live-tracked arrivals (populated once FlightAware has an active/filed flight plan)
- `GET /airports/{id}/flights/scheduled_arrivals` — schedule-based arrivals (published timetable, independent of live tracking)

`START_TIME`/`END_TIME` below are absolute timestamps you set directly — either
can be in the past or the future, independently.

Prerequisites:
- `.env` file in the project root with `FLIGHTAWARE_API_KEY=<your key>` (copy from `.env.example`)
- Dependencies installed in `.venv` (see project skill file at `.claude/skills/tartu-arrivals/SKILL.md`)

In [ ]:
%load_ext autoreload
%autoreload 2

In [17]:
import datetime as dt
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.flightaware_client import fetch_arrivals, fetch_scheduled_arrivals
from src.arrivals import to_dataframe, to_raw_dataframe, save_snapshot

## Parameters

Set `START_TIME` and `END_TIME` directly (both timezone-aware UTC). Examples:

```python
# Next 24 hours starting in 1 hour:
START_TIME = dt.datetime.now(dt.timezone.utc) + dt.timedelta(hours=1)
END_TIME = START_TIME + dt.timedelta(hours=24)

# Past 24 hours:
START_TIME = dt.datetime.now(dt.timezone.utc) - dt.timedelta(hours=24)
END_TIME = dt.datetime.now(dt.timezone.utc)

# A specific historical window:
START_TIME = dt.datetime(2026, 8, 1, 0, 0, tzinfo=dt.timezone.utc)
END_TIME = dt.datetime(2026, 8, 2, 0, 0, tzinfo=dt.timezone.utc)
```

In [28]:
AIRPORT_ICAO = "EETU"

START_TIME = dt.datetime(2026, 8, 8, 12, 0, tzinfo=dt.timezone.utc)
END_TIME = dt.datetime(2026, 8, 8, 19, 0, tzinfo=dt.timezone.utc)

## Endpoint 1: `flights/arrivals` (live-tracked)

In [19]:
raw_arrivals = fetch_arrivals(AIRPORT_ICAO, start=START_TIME, end=END_TIME)
raw_arrivals

{'arrivals': [], 'links': None, 'num_pages': 1}

Full response as a DataFrame — every field flattened, one row per flight.
Open this in Data Wrangler (right-click the variable in the Jupyter variable viewer,
or use the "Open in Data Wrangler" affordance on the cell output) to explore all
available columns before deciding what to keep.

In [20]:
df_arrivals_raw = to_raw_dataframe(raw_arrivals, key="arrivals")
df_arrivals_raw

""


Curated view (subset of columns, used for the comparison below).

In [21]:
df_arrivals = to_dataframe(raw_arrivals, key="arrivals")
df_arrivals

""


## Endpoint 2: `flights/scheduled_arrivals` (schedule-based)

In [29]:
raw_scheduled = fetch_scheduled_arrivals(AIRPORT_ICAO, start=START_TIME, end=END_TIME)
raw_scheduled

FlightAwareError: AeroAPI request failed (400): {"title":"Invalid argument","reason":"INVALID_ARGUMENT","detail":"Invalid end bound: time is too far in the future (limit: 2 days)","status":400}

Full response as a DataFrame — same idea as above, for exploring in Data Wrangler.

In [23]:
df_scheduled_raw = to_raw_dataframe(raw_scheduled, key="scheduled_arrivals")
df_scheduled_raw

,ident,ident_icao,ident_iata,actual_runway_off,actual_runway_on,fa_flight_id,operator,operator_icao,operator_iata,flight_number,...,origin.city,origin.airport_info_url,destination.code,destination.code_icao,destination.code_iata,destination.code_lid,destination.timezone,destination.name,destination.city,destination.airport_info_url
0,FIN1047,FIN1047,AY1047,None,None,FIN1047-1785907090-airline-984p,FIN,FIN,AY,1047,...,Vantaa,/airports/EFHK,EETU,EETU,TAY,None,Europe/Tallinn,Tartu,Tartu,/airports/EETU


Curated view (subset of columns, used for the comparison below).

In [24]:
df_scheduled = to_dataframe(raw_scheduled, key="scheduled_arrivals")
df_scheduled

,ident,origin,scheduled_in,estimated_in,actual_in,status
0,FIN1047,EFHK,2026-08-07 11:35:00+00:00,2026-08-07 11:35:00+00:00,NaT,Scheduled


## Compare

Row counts and which flight idents appear in one endpoint's response but not the other.

In [ ]:
arrivals_idents = set(df_arrivals["ident"]) if not df_arrivals.empty else set()
scheduled_idents = set(df_scheduled["ident"]) if not df_scheduled.empty else set()

print(f"arrivals: {len(arrivals_idents)} flights")
print(f"scheduled_arrivals: {len(scheduled_idents)} flights")
print()
print(f"only in arrivals: {arrivals_idents - scheduled_idents}")
print(f"only in scheduled_arrivals: {scheduled_idents - arrivals_idents}")
print(f"in both: {arrivals_idents & scheduled_idents}")

## Save snapshots (optional)

Saves both raw responses and flattened tables to `output/`, labeled by endpoint.

In [ ]:
arrivals_csv_path = save_snapshot(raw_arrivals, df_arrivals, AIRPORT_ICAO, label="arrivals")
scheduled_csv_path = save_snapshot(
    raw_scheduled, df_scheduled, AIRPORT_ICAO, label="scheduled_arrivals"
)
arrivals_csv_path, scheduled_csv_path